# FINN Dataflow Compiler: Analyse des Heatmap-Modells

Dieses Notebook implementiert den FINN-Builder-Flow für ein trainiertes ONNX-Modell. 
Ziel ist die Transformation des neuronalen Netzes in eine Streaming-Dataflow-Architektur, 
die Verifikation der numerischen Integrität und die Abschätzung des Ressourcenverbrauchs auf dem FPGA.

In [ ]:
import os
import shutil
import json
import numpy as np
import onnxruntime as ort
import finn.builder.build_dataflow as build
import finn.builder.build_dataflow_config as build_cfg
from finn.util.visualization import showInNetron
from qonnx.util.cleanup import cleanup as qonnx_cleanup

# --- 1. Konfiguration der Pfade ---
nb_dir = os.getcwd()
model_name = "untrained_heatmap_model.onnx" # Name ggf. anpassen
model_path = os.path.join(nb_dir, model_name)
output_dir = os.path.join(nb_dir, "output_heatmap_estimate")

# Prüfen der Eingabedatei
if not os.path.exists(model_path):
    raise FileNotFoundError(f"Modelldatei nicht gefunden: {model_path}")

print(f"Initialisiere Build-Prozess für: {model_path}")

## 2. Modell-Bereinigung (Cleanup)
Das Modell wird bereinigt, um sicherzustellen, dass alle Tensor-Dimensionen (Shapes) bekannt sind 
und die Graphenstruktur für den Compiler optimiert ist.

In [ ]:
clean_model_path = model_path.replace(".onnx", "_clean.onnx")
qonnx_cleanup(model_path, out_file=clean_model_path)
print(f"Bereinigtes Modell gespeichert: {clean_model_path}")

# Visualisierung des Eingangsmodells
showInNetron(clean_model_path)

## 3. Generierung von Referenzdaten für die Verifikation
Da das Modell trainierte Gewichte enthält, muss sichergestellt werden, dass die Hardware-Transformation 
das mathematische Verhalten nicht verändert. Wir erzeugen einen zufälligen Eingangsvektor 
und berechnen das erwartete Ergebnis mittels ONNX Runtime (Software-Emulation).

In [ ]:
# Session für ONNX Runtime erstellen
sess = ort.InferenceSession(clean_model_path)
input_name = sess.get_inputs()[0].name
input_shape = sess.get_inputs()[0].shape

# Zufälligen Input generieren (Float32)
# Hinweis: Shape-Handling für Batch-Size 1
if isinstance(input_shape[0], str): input_shape[0] = 1 
input_data = np.random.rand(*input_shape).astype(np.float32)

# Referenz-Output berechnen
expected_output = sess.run(None, {input_name: input_data})[0]

# Als .npy Dateien speichern (FINN erwartet diese Namen)
np.save("input.npy", input_data)
np.save("expected_output.npy", expected_output)
print("Referenzdaten (input.npy, expected_output.npy) generiert.")

## 4. Definition der Build-Konfiguration
Hier werden die Parameter für die Hardware-Synthese festgelegt. 
Der Fokus liegt auf einer Abschätzung (`ESTIMATE_REPORTS`), nicht auf der Bitfile-Generierung.

In [ ]:
# Verzeichnis bereinigen
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

cfg_estimates = build.DataflowBuildConfig(
    output_dir          = output_dir,
    mvau_wwidth_max     = 80,             # Parallelisierung der Matrix-Vektor-Einheiten
    target_fps          = 1000,           # Ziel-Durchsatz (Frames per Second)
    synth_clk_period_ns = 10.0,           # Ziel-Taktfrequenz (100 MHz)
    fpga_part           = "xc7z020clg400-1", # Ziel-Plattform (hier Pynq-Z1)
    
    # Standard-Schritte für reine Schätzung
    steps               = build_cfg.estimate_only_dataflow_steps,
    
    # Aktivierung der Verifikation für kritische Zwischenschritte
    verify_steps=[
        build_cfg.VerificationStepType.QONNX_TO_FINN_PYTHON,
        build_cfg.VerificationStepType.TIDY_UP_PYTHON,
        build_cfg.VerificationStepType.STREAMLINED_PYTHON,
    ],
    
    # Gewünschte Ausgabe-Artefakte
    generate_outputs=[
        build_cfg.DataflowOutputType.ESTIMATE_REPORTS,
    ]
)

In [ ]:
%%time
# Ausführen des Builder-Flows
build.build_dataflow_cfg(clean_model_path, cfg_estimates);

## 5. Analyse der Ergebnisse
Visualisierung des transformierten Hardware-Graphen und Ausgabe der geschätzten Ressourcen.

In [ ]:
# Visualisierung des Zwischenschritts "Convert to HW"
intermediate_model = os.path.join(output_dir, "intermediate_models", "step_convert_to_hw.onnx")
if os.path.exists(intermediate_model):
    print("Lade Hardware-Abstraktions-Graph...")
    showInNetron(intermediate_model)

# Auslesen des Ressourcen-Reports
res_file = os.path.join(output_dir, "report", "estimate_layer_resources.json")
if os.path.exists(res_file):
    with open(res_file, 'r') as f:
        res_data = json.load(f)
    print("\n--- Geschätzter Ressourcenverbrauch (Total) ---")
    print(json.dumps(res_data["total"], indent=2))
else:
    print("Keine Ressourcenschätzung gefunden.")

# Auslesen der Verification Results
verif_dir = os.path.join(output_dir, "verification_output")
if os.path.exists(verif_dir):
    print("\n--- Verifikations-Status ---")
    print("Geprüfte Schritte:")
    # Auflisten der .npy Dateien, die auf _SUCCESS oder _FAIL enden
    files = [f for f in os.listdir(verif_dir) if "SUCCESS" in f or "FAIL" in f]
    for f in files:
        print(f" - {f}")